# Gaussian Splatting with [gsplat](https://docs.gsplat.studio/main/) library

`gsplat` is a PyTorch-based library for training and rendering 3D Gaussian Splatting models.
It provides efficient CUDA implementations for rasterization, optimization, densification and pruning, so we can train a Gaussian scene representation directly from the camera poses, intrinsics and images produced by COLMAP.

### Installation

gsplat is built from source here, so its CUDA kernels are compiled against the exact PyTorch and CUDA versions in use. Tested setup: Ubuntu, NVIDIA driver 580 (supports CUDA up to 13.0), RTX 3050 Laptop GPU (compute capability 8.6), CUDA toolkit 12.9, conda environment with Python 3.12.

Check that the driver and the CUDA toolkit are available:

```bash
nvidia-smi                                   # driver version and max supported CUDA version
/usr/local/cuda-12.9/bin/nvcc --version      # CUDA toolkit used to compile gsplat
```

Install PyTorch with a CUDA 12.x build, which matches the major version of the toolkit:

```bash
pip install torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/cu128
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"   # 2.9.1+cu128 True
```

Install the build tools, then clone gsplat with its submodules:

```bash
pip install ninja setuptools wheel
git clone --recursive https://github.com/nerfstudio-project/gsplat.git ~/Apps/gsplat
cd ~/Apps/gsplat
git submodule update --init --recursive      # in case the clone was not recursive
```

Build gsplat against the installed PyTorch:

```bash
# TORCH_CUDA_ARCH_LIST: compile only for the RTX 3050 (Ampere, 8.6)
# MAX_JOBS: limit parallel compile jobs to avoid running out of RAM
# --no-build-isolation: compile against the installed torch, not a fresh one from PyPI
TORCH_CUDA_ARCH_LIST="8.6" MAX_JOBS=2 pip install -e . --no-build-isolation
python -c "import gsplat; from gsplat import rasterization; print(gsplat.__version__)"
```

Install the dependencies of the example trainer:

```bash
cd examples
pip install -r requirements.txt --no-build-isolation
```

If PyTorch reports `CUDA unknown error` after the laptop was suspended, reload the NVIDIA UVM kernel module (or reboot):

```bash
sudo rmmod nvidia_uvm
sudo modprobe nvidia_uvm
```

## Image resolution

The 4 GB GPU can't train on full-resolution images, so training uses
`--data_factor 8` (1/8 width and height). gsplat's COLMAP parser requires an
`images_8/` folder to exist. When it contains JPGs, the parser downscales the
original `images/` by 8× itself, caches the result in `images_8_png/`, and
rescales the intrinsics to match. A symlink is enough to satisfy the check:

```bash
cd colmap_results/0_undistorted
ln -s images images_8
```

In [2]:
%%bash

# Using simple_trainer.
cd ~/Apps/gsplat/examples

python simple_trainer.py default \
    --data_dir ~/code/MA/gs-intro/colmap_results/0_undistorted \
    --data_factor 8 \
    --packed \
    --save_ply \
    --result_dir ~/code/MA/gs-intro/gsplat_results/tomatoes_antialiased \
    --max_steps 5000 --eval_steps 2500 5000 --save_steps 2500 5000

[Parser] 166 images, taken by 166 cameras.
Downscaling images by 8x from /home/boldi/code/MA/gs-intro/colmap_results/0_undistorted/images to /home/boldi/code/MA/gs-intro/colmap_results/0_undistorted/images_8_png.


Loading EXIF exposure: 100%|██████████| 166/166 [00:00<00:00, 41852.28it/s]


[Parser] No valid EXIF exposure data found in any image.
Scene scale: 1.529745695083774
Model initialized. Number of GS: 25925
╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯


loss=0.112| sh degree=0| :  12%|█▏        | 598/5000 [00:26<03:10, 23.15it/s]

Step 600: 1296 GSs duplicated, 777 GSs split. Now having 27998 GSs.
Step 600: 19 GSs pruned. Now having 27979 GSs.


...

Training complete, disable training tab.
Viewer running... Ctrl+C to exit.


Traceback (most recent call last):
  File "/home/boldi/Apps/gsplat/examples/simple_trainer.py", line 1617, in <module>
    cli(main, cfg, verbose=True)
  File "/home/boldi/Apps/gsplat/gsplat/distributed.py", line 375, in cli
    return _distributed_worker(0, 1, fn=fn, args=args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/boldi/Apps/gsplat/gsplat/distributed.py", line 310, in _distributed_worker
    fn(local_rank, world_rank, world_size, args)
  File "/home/boldi/Apps/gsplat/examples/simple_trainer.py", line 1558, in main
    time.sleep(1000000)
KeyboardInterrupt


(viser) Server stopped


TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType

In [ ]:
%%bash

# Using simple_trainer mcmc.
cd ~/Apps/gsplat/examples

python simple_trainer.py mcmc \
    --data_dir ~/code/MA/gs-intro/colmap_results/0_undistorted \
    --data_factor 8 \
    --packed \
    --save_ply \
    --strategy.cap-max 300000 \
    --result_dir ~/code/MA/gs-intro/gsplat_results/tomatoes_mcmc \
    --max_steps 5000 --eval_steps 2500 5000 --save_steps 2500 5000

## gsplat Training Output

Each run writes one result folder, e.g. `gsplat_results/tomatoes/`:

| Folder | Content |
| --- | --- |
| `ckpts/` | `ckpt_<step>_rank0.pt`: the trained model, a PyTorch dict with every Gaussian parameter (means, quats, scales, opacities, SH coefficients) plus the step and config |
| `ply/` | `point_cloud_<step>.ply`: the same Gaussians in the standard 3DGS PLY format, written only with `--save_ply` |
| `renders/` | `val_step<step>_XXXX.png`: held-out views, ground truth and render side by side |
| `stats/` | `val_step<step>.json` with PSNR, SSIM, LPIPS and the Gaussian count; `train_step*.json` with timing and memory |
| `tb/` + `cfg.yml` | TensorBoard logs and the exact config the run used |
| `videos/` | `traj_<step>.mp4`: a fly-through along an interpolated camera path |

### Inspect the model interactively

In [ ]:
%%bash

cd ~/Apps/gsplat/examples
python simple_viewer.py \
    --ckpt ~/code/MA/gs-intro/gsplat_results/tomatoes/ckpts/ckpt_4999_rank0.pt \
    --port 8080

### Load the parameters directly

In [ ]:
import torch

ckpt = torch.load("gsplat_results/tomatoes/ckpts/ckpt_4999_rank0.pt", map_location="cpu")
splats = ckpt["splats"]
print({k: tuple(v.shape) for k, v in splats.items()})

### Read the baseline numbers with TensorBoard (tb)

In [ ]:
%%bash

tensorboard --logdir gsplat_results/tomatoes/tb